# 실습 6 · CNN을 쌓아 내 사진을 분류하기

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

**⏱ 75분** · 6주차 실습 — 콘볼루션·ReLU·풀링 블록을 쌓아 컬러 사진 열 종류를 분류하고, 직접 찍은 사진으로 시험하기

## 🎯 이번 시간의 질문

콘볼루션 층, ReLU, 풀링을 하나씩 확인한 뒤 이 셋을 한 블록으로 묶어 세 번 쌓고, 컬러 사진 열 종류를 분류합니다.

> **블록을 쌓은 CNN은 내가 찍은 사진을 맞힐까요?**

CIFAR-10 사진 4만 장으로 학습시킨 뒤, 휴대폰으로 찍은 사진을 올려 판별합니다.

**이번 시간에 반복할 패턴:** `nn.Conv2d`의 출력 모양과 파라미터 수 · `nn.ReLU`·`nn.MaxPool2d` · 블록을 쌓으며 모양 찍기 · `nn.Module`로 특성 학습부와 분류부 조립 · `Normalize`로 채널별 표준화 · `.to(device)` · 사진 한 장을 학습 때와 같은 전처리로 바꿔 예측하기.

**대응 이론:** [Ch07 CNN의 구조와 아키텍처](https://ralbu85.github.io/lecture_deeplearning/chapters/ch07.html)

**Colab 설정:** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 시작합니다. 5절의 학습이 GPU에서 2~3분 걸립니다.

| 표시 | 뜻 |
|---|---|
| ▶ | 그대로 실행합니다 |
| ✍️ | 보여 준 코드를 손으로 타이핑해서 실행합니다. 빈칸 `___`은 채웁니다 |
| 📝 | 과제. 힌트를 보고 스스로 풉니다 |

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요.
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image, ImageOps

torch.manual_seed(42)
print("준비 완료!")

`준비 완료!`가 나오면 준비가 끝난 것입니다.

---

## 1. 콘볼루션 층

`nn.Conv2d`는 작은 커널을 사진 위로 한 칸씩 옮기며 곱하고 더해 특성 맵을 만드는 층입니다. `nn.Conv2d(입력 채널 수, 커널 수, kernel_size=커널 크기)`로 만듭니다. 컬러 사진은 R, G, B 세 장이므로 입력 채널 수는 3입니다. 3×3 커널 32개짜리 층을 만들고, 32×32 컬러 사진 한 장 모양의 입력 `(1, 3, 32, 32)`를 넣습니다. 층은 `(사진 수, 채널, 세로, 가로)` 모양을 받습니다. 입력 값은 `torch.randint(-9, 10, 모양)`으로 채웁니다. −9부터 9까지의 정수를 무작위로 뽑아 그 모양의 텐서를 만드는 함수입니다. 층은 실수를 받으므로 `.float()`로 실수로 바꾸고, 앞에 `torch.manual_seed(42)`를 두어 누가 실행해도 같은 값이 나오게 합니다. 이 실습의 예시 입력은 모두 이 방법으로 만듭니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
torch.manual_seed(42)
x = torch.randint(-9, 10, (1, 3, 32, 32)).float()
conv = nn.Conv2d(3, 32, kernel_size=3)
h = conv(x)
print(h.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `torch.Size([1, 32, 30, 30])`입니다. 커널 하나가 특성 맵 한 장을 만들므로 커널 32개에서 특성 맵 32장이 나와 채널이 3에서 32가 됩니다. 3×3 커널은 가장자리 한 칸에서는 사진 밖으로 나가 계산할 수 없으므로 한 변이 32에서 30으로 줄어듭니다.

테두리에 0을 한 겹 두르는 `padding=1`을 주면 가장자리에서도 계산할 수 있어 크기가 그대로 남습니다. 입력은 위에서 만든 `x`를 그대로 넣습니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
h = conv(x)
print(h.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `torch.Size([1, 32, 32, 32])`로, 크기 32×32가 그대로 남습니다.

층이 학습하는 값은 커널의 가중치와 편향입니다. 커널 하나는 3×3 크기가 입력 채널 3장만큼 있어 가중치가 3×3×3 = 27개이고, 편향이 1개 붙습니다. 커널이 32개이므로 이것의 32배입니다. `conv.weight.shape`로 확인합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(conv.weight.shape, conv.bias.shape)
print(3 * 3 * 3 * 32 + 32)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

가중치는 `torch.Size([32, 3, 3, 3])`, 편향은 `torch.Size([32])`이고, 파라미터는 **896**개입니다. `weight.shape`의 네 숫자는 (커널 수, 입력 채널 수, 커널 세로, 커널 가로)입니다. 파라미터 수에 사진 크기 32×32는 들어가지 않습니다. 커널 하나가 사진의 모든 위치를 돌며 같은 가중치를 쓰기 때문입니다.

---

## 2. ReLU와 풀링을 층으로 쓴다

콘볼루션 층 뒤에는 ReLU와 풀링이 붙습니다. `nn.ReLU`는 음수를 0으로 바꾸고 양수는 그대로 두는 층입니다. 무작위 정수로 채운 3×3 행렬에 적용합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
relu = nn.ReLU()
torch.manual_seed(42)
x = torch.randint(-9, 10, (3, 3)).float()
print(x)
print(relu(x))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

음수인 −7, −3, −4 세 칸만 0이 되고 나머지는 그대로입니다.

`nn.MaxPool2d(2)`는 Max 풀링 층입니다. 2×2 창을 두 칸씩 옮기며 창마다 가장 큰 값 하나만 남깁니다. 층은 `(사진 수, 채널, 세로, 가로)` 모양을 받으므로 입력은 사진 1장, 채널 1장인 4×4 특성 맵 `(1, 1, 4, 4)`로 만듭니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
pool = nn.MaxPool2d(2)
torch.manual_seed(42)
x = torch.randint(-9, 10, (1, 1, 4, 4)).float()
print(x)
print(pool(x))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `[[6., 6.], [4., 8.]]`입니다. 창이 네 개이므로 2×2가 나오고, 값은 네 창의 최댓값입니다.

| 창 | 값 네 개 | 최댓값 |
|---|---|---|
| 왼쪽 위 | 6, 4, −7, 5 | 6 |
| 오른쪽 위 | 2, 5, 6, −3 | 6 |
| 왼쪽 아래 | −4, 4, −6, −6 | 4 |
| 오른쪽 아래 | −7, 6, 8, −2 | 8 |

채널이 여러 장이면 어떻게 되는지 봅니다. 크기 14×14, 채널 6장인 입력을 넣고, 풀링 층에 학습할 파라미터가 있는지 꺼내 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
torch.manual_seed(42)
x = torch.randint(-9, 10, (1, 6, 14, 14)).float()
print(pool(x).shape)
print(list(pool.parameters()))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `torch.Size([1, 6, 7, 7])`과 빈 목록 `[]`입니다. 풀링은 채널마다 따로 적용되어 크기만 절반이 되고 채널 6은 그대로입니다. 최댓값을 고르는 규칙뿐이라 학습할 파라미터가 없습니다. `nn.ReLU`도 마찬가지입니다.

---

## 3. 블록을 쌓고 모양을 따라간다

3×3 콘볼루션 → ReLU → 2×2 Max 풀링을 한 블록으로 보고 세 번 쌓습니다. 커널 수는 32, 64, 128로 늘립니다. 두 번째 블록의 입력 채널 수는 첫 블록의 커널 수 32이고, 세 번째 블록은 64입니다. 콘볼루션에는 1절처럼 `padding=1`을 주므로 크기는 풀링에서만 절반으로 줄어듭니다. 32×32 컬러 사진 한 장 모양의 입력 `(1, 3, 32, 32)`를 넣고 층을 하나씩 통과시키며 모양을 찍습니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
features = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)

torch.manual_seed(42)
x = torch.randint(-9, 10, (1, 3, 32, 32)).float()
h = x
for layer in features:
    h = layer(h)
    print(layer.__class__.__name__, tuple(h.shape))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

마지막 줄은 `MaxPool2d (1, 128, 4, 4)`입니다. 크기는 풀링마다 32 → 16 → 8 → 4로 줄고, 채널은 3 → 32 → 64 → 128로 늘어납니다. 층을 지날수록 특성 맵은 작아지고 장수는 많아집니다.

뒤에 분류부를 붙입니다. 128×4×4를 펼쳐 은닉 256 노드에 넣고, 열 클래스의 점수를 냅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
    nn.Linear(256, 10),
)
print(classifier(h).shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `torch.Size([1, 10])`으로, 열 클래스의 점수 10개입니다.

두 부분을 모델 하나로 조립합니다. `nn.Module`을 물려받는 클래스를 만들고, `__init__`에는 층을, `forward`에는 계산 순서를 적습니다. `super().__init__()`은 `nn.Module`의 준비 작업을 먼저 실행하는 줄입니다. 실습 7부터 나오는 모델이 모두 이 형태입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        h = self.features(x)
        return self.classifier(h)

torch.manual_seed(42)
model = CNN()
x = torch.randint(-9, 10, (2, 3, 32, 32)).float()
print(model(x).shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

사진 두 장을 넣었으므로 출력은 `torch.Size([2, 10])`입니다. `self.features`는 사진에서 특성 맵을 뽑는 특성 학습부, `self.classifier`는 그 특성으로 종류를 가르는 분류부입니다. `model(x)`를 부르면 `forward`가 실행됩니다.

---

## 4. 컬러 사진 데이터

CIFAR-10은 32×32 컬러 사진 6만 장이고 열 종류로 나뉩니다. 실습 4에서 열마다 훈련 데이터의 평균과 표준편차로 표준화했듯이, 사진도 **채널마다** 표준화합니다. 아래 숫자는 훈련 사진 5만 장의 R, G, B 채널별 평균과 표준편차입니다. `transforms.Compose`는 나열한 변환을 차례로 적용합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

In [ ]:
# ▶ 그대로 실행하세요 — CIFAR-10을 내려받습니다(약 170MB). 원본 서버가 느려 수업용 미러를 씁니다.
datasets.CIFAR10.url = 'https://github.com/ralbu85/lecture_deeplearning_labs/releases/download/data-v1/cifar-10-python.tar.gz'
tr_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
te_full = datasets.CIFAR10(root='./data', train=False, download=True, transform=tf)
print(tr_full.classes)

`airplane`부터 `truck`까지 열 개의 이름이 나옵니다. 정답 번호 0~9가 이 순서입니다.

CIFAR-10에는 훈련 5만 장과 테스트 1만 장만 있고 검증 세트가 따로 없습니다. 그래서 훈련 5만 장을 훈련 4만 장과 검증 1만 장으로 나눕니다. 사진을 직접 나누지 않고 사진 번호를 나눕니다. 사진 번호는 데이터셋에서 몇 번째 사진인지를 나타내는 0~49999의 정수이고, `tr_full[0]`이 0번 사진입니다. 이 번호를 `train_test_split`으로 8 : 2로 나눕니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
tr_idx, va_idx = train_test_split(range(50000), test_size=0.2, random_state=42)
print(len(tr_idx), len(va_idx))
print(tr_idx[:5])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

훈련 번호가 **40000**개, 검증 번호가 **10000**개입니다. 앞 다섯 개가 `[39087, 30893, 45278, 16398, 13653]`처럼 섞여 있어, 훈련과 검증에 열 종류가 고르게 들어갑니다.

`Subset(데이터셋, 번호 리스트)`는 데이터셋에서 리스트에 든 번호의 사진만 골라 낸 새 데이터셋입니다. 번호 리스트는 위에서 만든 `tr_idx`처럼 사진 번호를 담은 리스트입니다. `Subset(tr_full, tr_idx)`는 `tr_full`의 39087번, 30893번, … 사진 4만 장으로 된 데이터셋입니다. 사진을 복사하지 않고 번호만 기억했다가, 꺼낼 때 `tr_full`에서 그 번호의 사진을 가져옵니다. 훈련 번호로 훈련 세트를, 검증 번호로 검증 세트를 만듭니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
train_ds = Subset(tr_full, tr_idx)
val_ds = Subset(tr_full, va_idx)
print(len(train_ds), len(val_ds))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

훈련 세트는 **40000**장, 검증 세트는 **10000**장입니다.

`train_ds`의 0번 사진은 원래 데이터셋의 `tr_idx[0]`번, 곧 39087번 사진입니다. 두 쪽에서 꺼내 정답 번호를 비교합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
x0, y0 = train_ds[0]
x1, y1 = tr_full[tr_idx[0]]
print(y0, y1, tr_full.classes[y0])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

둘 다 **6**(`frog`)입니다. `Subset`의 0번이 원래 데이터셋의 39087번을 가리킵니다.

세 데이터셋을 로더에 넣습니다. 테스트는 CIFAR-10이 따로 둔 1만 장을 그대로 씁니다. 훈련 로더만 `shuffle=True`로 에폭마다 순서를 섞습니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(te_full, batch_size=256)
print(len(train_loader), len(val_loader), len(test_loader))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

훈련은 **313**배치입니다. 4만 장을 128장씩 나누면 312배치에 64장이 남아 313배치가 됩니다. 검증과 테스트는 256장씩 **40**배치입니다.

배치 하나를 꺼내 모양과 표준화 결과를 봅니다. `xb[:, 0]`은 배치의 R 채널입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
xb, yb = next(iter(train_loader))
print(tuple(xb.shape), yb[:8])
print(xb[:, 0].mean().item(), xb[:, 0].std().item())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

배치는 `(128, 3, 32, 32)`입니다. 컬러라서 채널이 3입니다. R 채널의 평균은 0 근처, 표준편차는 1 근처입니다. `Normalize`가 채널마다 평균을 빼고 표준편차로 나누었기 때문입니다.

---

## 5. GPU에서 학습시킨다

`device`는 계산할 장치입니다. GPU가 있으면 `cuda`, 없으면 `cpu`입니다. 모델을 `.to(device)`로 그 장치에 옮깁니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

Colab에서 GPU 런타임이면 `cuda`가 나옵니다.

모델과 데이터는 같은 장치에 있어야 계산됩니다. 학습 함수는 배치마다 다섯 줄(기울기 지우기 → 예측 → 손실 → 기울기 계산 → 갱신)을 실행하고, 그 앞에 배치를 장치로 옮기는 줄이 들어갑니다. 빈칸을 채웁니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = xb.to(___), yb.to(___)      # 배치를 모델과 같은 장치로
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n

평가 함수는 가중치를 바꾸지 않고 손실과 맞힌 수를 셉니다. 여기에도 배치를 장치로 옮기는 줄이 들어갑니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def evaluate(model, loader, criterion):
    model.eval()
    total = 0.0
    correct = 0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
    return total / n, correct / n
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

손실과 옵티마이저를 만들고, 학습 전의 검증 손실과 정확도를 봅니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(evaluate(model, val_loader, criterion))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

손실은 약 **2.30**, 정확도는 **0.1** 안팎입니다. 열 종류를 찍는 수준입니다.

10 에폭을 돌리며 에폭마다 검증 손실과 정확도를 찍습니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
for epoch in range(10):
    train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(epoch, round(val_loss, 4), round(val_acc, 4))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

검증 정확도는 첫 에폭에 약 0.58이고 0.75 근처에서 더 오르지 않습니다. 검증 손실은 여섯 번째 에폭 무렵 가장 낮고 그 뒤로 다시 올라갑니다. 모델이 훈련 사진에만 맞춰지는 과적합의 신호입니다.

테스트 세트로 한 번 평가합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(test_acc)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

테스트 정확도는 약 **0.75**입니다. 처음 보는 사진 열 장 중 일곱 장 이상을 맞힙니다.

---

## 6. 내 사진을 판별한다

모델은 32×32로 표준화된 사진만 받습니다. 휴대폰 사진도 같은 모양으로 바꿉니다. `Resize(32)`는 짧은 변을 32로 줄이고, `CenterCrop(32)`는 가운데 32×32를 잘라 냅니다. 그 뒤는 학습 때와 같은 `ToTensor`와 `Normalize`입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
photo_tf = transforms.Compose([
    transforms.Resize(32),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

예시 사진으로 먼저 해 봅니다. scikit-image에 들어 있는 고양이 사진입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 예시 사진 불러오기
from skimage import data
photo = Image.fromarray(data.chelsea())
plt.imshow(photo)
plt.axis('off')
plt.show()

사진을 변환하고 모양을 봅니다. `photo.size`는 PIL 사진의 (너비, 높이)입니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
x = photo_tf(photo)
print(photo.size, tuple(x.shape))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

출력은 `(451, 300) (3, 32, 32)`입니다. 451×300 사진이 모델이 받는 `(3, 32, 32)`가 되었습니다.

모델에 들어가는 32×32 사진을 그려 봅니다. 표준화하기 전 단계까지만 적용한 그림입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 모델이 보는 32×32 사진
small = transforms.CenterCrop(32)(transforms.Resize(32)(photo))
plt.imshow(small)
plt.axis('off')
plt.show()

학습에 쓴 CIFAR-10 사진과 같은 해상도입니다. 수염과 털의 결은 사라지고 얼굴의 윤곽과 색만 남습니다.

예측합니다. 모델은 배치를 받으므로 `unsqueeze(0)`으로 `(1, 3, 32, 32)`를 만들고, 모델과 같은 장치로 옮깁니다. 점수 10개를 `softmax`로 확률로 바꿉니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
model.eval()
with torch.no_grad():
    out = model(x.unsqueeze(0).to(device))
prob = torch.softmax(out, dim=1)[0]
print(tr_full.classes[prob.argmax().item()], prob.max().item())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

예측한 종류와 그 확률이 나옵니다. 같은 코드로 학습을 다시 하면 이 사진의 판별은 `cat`, `dog`, `frog` 사이에서 바뀝니다. 테스트 정확도는 매번 0.75 근처인데 사진 한 장의 판별은 흔들립니다.

사진 한 장을 받아 예측 이름과 확률을 돌려주는 함수로 조립합니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def predict(photo):
    x = photo_tf(photo).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1)[0]
    return tr_full.classes[prob.argmax().item()], prob.max().item()

print(predict(photo))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

바로 위와 같은 결과가 나옵니다.

이제 내 사진을 올립니다. 아래 셀은 **Colab에서만** 실행됩니다. 실행하면 파일 선택 버튼이 나타나고, 사진(JPG, PNG)을 여러 장 고를 수 있습니다. 사람은 찍지 않습니다.

In [ ]:
# ▶ 그대로 실행하세요 — (Colab) 내 사진 올리기
from google.colab import files
uploaded = files.upload()

올린 사진마다 `predict`를 실행합니다. `ImageOps.exif_transpose`는 휴대폰 사진에 저장된 회전 정보를 반영해 사진을 바로 세웁니다.


**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
for name in uploaded:
    photo = ImageOps.exif_transpose(Image.open(name)).convert('RGB')
    print(name, predict(photo))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요

사진 이름 옆에 예측한 종류와 확률이 나옵니다. 물체가 사진 가운데에 크게 찍힌 사진일수록 잘 맞힙니다. `CenterCrop`이 가운데만 남기기 때문입니다.

---

## 7. 실습 과제 (10분)

### 📝 과제 1 · 내 사진 다섯 장

CIFAR-10의 열 종류(`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`) 가운데 주변에서 찍을 수 있는 것을 다섯 장 찍어 올리고, 판별 결과를 표에 적습니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 6절의 업로드 셀을 다시 실행해 사진을 올리고, for 반복문으로 사진마다 predict(photo)를 실행합니다.

| 사진 | 실제 종류 | 예측 | 확률 |
|---|---|---|---|
| 1 | ✏️ | ✏️ | ✏️ |
| 2 | ✏️ | ✏️ | ✏️ |
| 3 | ✏️ | ✏️ | ✏️ |
| 4 | ✏️ | ✏️ | ✏️ |
| 5 | ✏️ | ✏️ | ✏️ |


### 📝 과제 2 · 틀린 사진 들여다보기

틀린 사진 한 장을 골라 모델이 보는 32×32 사진을 그리고, 열 종류의 확률을 모두 출력합니다. 정답 종류의 확률은 몇 번째로 높은지 적습니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 6절의 small = transforms.CenterCrop(32)(transforms.Resize(32)(photo))로 그림을 그립니다. 확률은 predict 함수 안의 세 줄로 prob을 구한 뒤, for name, p in zip(tr_full.classes, prob): 로 하나씩 출력합니다.

✏️ **정답 종류의 순위와 확률:**


### 📝 과제 3 · 생각해 보기

테스트 정확도는 0.75인데 내 사진은 그보다 자주 틀릴 수 있습니다. 학습한 사진과 내 사진은 무엇이 다를까요? 한두 문장으로 적습니다.

✏️ **나의 답:**


---

## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 콘볼루션 층 | `nn.Conv2d(3, 32, kernel_size=3, padding=1)`. 출력 `(1, 32, 32, 32)`, 파라미터 3×3×3×32 + 32 = 896 |
| ReLU·풀링 층 | `nn.ReLU()`, `nn.MaxPool2d(2)`. 풀링은 크기 절반, 채널 유지. 둘 다 파라미터 없음 |
| 블록 쌓기 | 3×3 콘볼루션(`padding=1`) → ReLU → 풀링을 세 번. 32 → 16 → 8 → 4, 채널 3 → 32 → 64 → 128 |
| 모델 조립 | `class CNN(nn.Module)`. `__init__`에 특성 학습부와 분류부, `forward`에 계산 순서 |
| 훈련·검증 나누기 | 번호 0~49999를 `train_test_split`으로 나누고 `Subset(tr_full, tr_idx)`로 그 번호의 사진만 골라 냄 |
| 채널별 표준화 | `transforms.Normalize(mean, std)`. 훈련 사진의 채널별 평균과 표준편차 |
| GPU | `device`, `model.to(device)`, 배치마다 `xb.to(device)` |
| 학습 결과 | 10 에폭, 테스트 정확도 약 0.75 |
| 내 사진 판별 | `Resize(32)` → `CenterCrop(32)` → `ToTensor` → `Normalize` → `unsqueeze(0)` → `softmax` |

**다음:** [실습 7](https://ralbu85.github.io/lecture_deeplearning/labs/lab07.html)에서는 이미 학습된 모델을 가져와 적은 사진으로 학습시킵니다.